In [ ]:
from pynq import Overlay
from pynq import allocate
import numpy as np
import time

In [ ]:
ol = Overlay("cnn_dma_3.bit")

In [ ]:
# Connect to the GPIO FIRST
gpio = ol.axi_gpio_0
gpio.write(0, 0x1)
time.sleep(0.1) 

In [ ]:
dma = ol.axi_dma_0

In [ ]:
def hardware_reset():
    gpio.write(0, 0x0) 
    time.sleep(0.01)
    gpio.write(0, 0x1)
    time.sleep(0.01)

In [ ]:
kernel = np.array([
    [-1, -1, -1, -1, -1],
    [-1, -2, -2, -2, -1],
    [-1, -2, 48, -2, -1],
    [-1, -2, -2, -2, -1],
    [-1, -1, -1, -1, -1]
], dtype=np.int32)

In [ ]:
def sw_conv(img):
    out = np.zeros((24, 24), dtype=np.int32)
    for i in range(24):
        for j in range(24):
            out[i, j] = np.sum(img[i:i+5, j:j+5] * kernel)
    return out

In [ ]:
inp = allocate(shape=(784,), dtype=np.int32)
out = allocate(shape=(576,), dtype=np.int32)

Preliminary tests.

In [ ]:
for i in range(5):
    img = np.arange(784, dtype=np.int32) % 256  
    img = img.reshape(28, 28)
    
    # reset the Conv2D IP and DMA before every run
    hardware_reset()
    dma.sendchannel.start()
    dma.recvchannel.start()
    
    # Load data into DMA memory
    inp[:] = img.flatten()
    
    t1 = time.time()
    
    # Fire the DMA transfers
    dma.sendchannel.transfer(inp)
    dma.recvchannel.transfer(out)
    dma.sendchannel.wait()
    print(f"Test {i+1}: Send complete...") 
    
    dma.recvchannel.wait() 
    print(f"Test {i+1}: Receive complete!") 
    
    hw_time = (time.time() - t1) * 1000
    hw_out = np.array(out).reshape(24, 24)
    
    # Run Software Reference & Compare
    t2 = time.time()
    sw_out = sw_conv(img)
    sw_time = (time.time() - t2) * 1000
    
    match = np.array_equal(hw_out, sw_out)
    
    print(f"  HW Time: {hw_time:.2f} ms | SW Time: {sw_time:.2f} ms")
    print(f"  Data Match: {match}\n")

Test 1: Send complete...
Test 1: Receive complete!
  HW Time: 4.52 ms | SW Time: 68.21 ms
  Data Match: True

Test 2: Send complete...
Test 2: Receive complete!
  HW Time: 2.96 ms | SW Time: 66.38 ms
  Data Match: True

Test 3: Send complete...
Test 3: Receive complete!
  HW Time: 2.97 ms | SW Time: 85.10 ms
  Data Match: True

Test 4: Send complete...
Test 4: Receive complete!
  HW Time: 2.97 ms | SW Time: 66.18 ms
  Data Match: True

Test 5: Send complete...
Test 5: Receive complete!
  HW Time: 2.96 ms | SW Time: 66.37 ms
  Data Match: True
